In [10]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

# --------------------------------------------------
# 1. Load dataset
# --------------------------------------------------
df = pd.read_csv("../data/merged_dataset_organized_47photic.csv")

# --------------------------------------------------
# 1B. Extract metadata from NaN-depth rows (one per cast)
# --------------------------------------------------
metadata_cols = ["CAST_COUNT", "Cruise_ID", "Cruz_Sta", "Cast_ID", "Sta_ID",
                 "Distance", "Date", "Time", "Lat_Dec", "Lon_Dec",
                 "Ac_Line", "Bottom_D", "Secchi", "IntChl", "IntC14",
                 "TimeZone", "Visibility"]

metadata = (
    df[df["DEPTH"].isna()][metadata_cols]
    .drop_duplicates("CAST_COUNT")
)

# --------------------------------------------------
# 2. Define baseline depth
# --------------------------------------------------
BASELINE_DEPTH = 2

# --------------------------------------------------
# 3. Create summed variables (2m to photic depth per cast)
# --------------------------------------------------
# First get photic depth per cast
photic_depths = df[df["PHOTIC_ZONE"] == True][["CAST_COUNT", "DEPTH"]].rename(
    columns={"DEPTH": "PHOTIC_DEPTH"}
).drop_duplicates("CAST_COUNT")

# Join photic depth onto full dataframe
df_with_photic = df.merge(photic_depths, on="CAST_COUNT", how="left")

# Filter to baseline → photic depth range per cast
df_range = df_with_photic[
    (df_with_photic["DEPTH"] >= BASELINE_DEPTH) &
    (df_with_photic["DEPTH"] <= df_with_photic["PHOTIC_DEPTH"])
]

summed = df_range.groupby("CAST_COUNT").agg(
    XMISS_SUMMED=("XMISS", "sum"),
    CHL_A_SUMMED=("CHL_A", "sum")
).reset_index()

# --------------------------------------------------
# 4. Keep only photic_zone == TRUE rows (one row per cast)
# --------------------------------------------------
df_photic = df[df["PHOTIC_ZONE"] == True].copy()

# --------------------------------------------------
# 5. Merge summed variables onto photic-zone dataframe
# --------------------------------------------------
df_final = df_photic.merge(summed, on="CAST_COUNT", how="left")

# Merge metadata (drop sparse versions first, then bring in clean ones)
df_final = df_final.drop(columns=[c for c in metadata_cols[1:] if c in df_final.columns])
df_final = df_final.merge(metadata, on="CAST_COUNT", how="left")

# --------------------------------------------------
# 6. Impute ESTCHL_STACORR (median imputation)
# --------------------------------------------------
median_estchl = df_final["ESTCHL_STACORR"].median()
df_final["ESTCHL_STACORR"] = df_final["ESTCHL_STACORR"].fillna(median_estchl)

# --------------------------------------------------
# 7A. Two-point Beer-Lambert K_PAR (original formula)
# --------------------------------------------------

# Baseline PAR at 2m
baseline_par = df[df["DEPTH"] == BASELINE_DEPTH][
    ["CAST_COUNT", "PAR"]
].rename(columns={"PAR": "PAR_BASELINE"}).drop_duplicates("CAST_COUNT")

df_final = df_final.merge(baseline_par, on="CAST_COUNT", how="left")

df_final["K_PAR"] = -np.log(
    df_final["PAR"] / df_final["PAR_BASELINE"]
) / df_final["DEPTH"]

df_final = df_final.drop(columns=["PAR_BASELINE"])  # remove intermediate column


# --------------------------------------------------
# 7B. Regression slope-based K_PAR_SLOPE
# --------------------------------------------------

kpar_slope_results = []

for cast_id, cast_df in df.groupby("CAST_COUNT"):

    # Get photic depth
    photic_row = cast_df[cast_df["PHOTIC_ZONE"] == True]
    if photic_row.empty:
        continue

    photic_depth = photic_row["DEPTH"].values[0]

    # Subset depths between 2m and photic depth
    subset = cast_df[
        (cast_df["DEPTH"] >= BASELINE_DEPTH) &
        (cast_df["DEPTH"] <= photic_depth)
    ].copy()

    subset = subset[subset["PAR"] > 0]

    if len(subset) < 2:
        continue

    subset["LOG_PAR"] = np.log(subset["PAR"])

    X = subset[["DEPTH"]].values
    y = subset["LOG_PAR"].values

    model = LinearRegression()
    model.fit(X, y)

    slope = model.coef_[0]

    kpar_slope_results.append({
        "CAST_COUNT": cast_id,
        "K_PAR_SLOPE": -slope
    })

kpar_slope_df = pd.DataFrame(kpar_slope_results)

df_final = df_final.merge(kpar_slope_df, on="CAST_COUNT", how="left")

# --------------------------------------------------
# 8. Save final dataset
# --------------------------------------------------
df_final.to_parquet("../data/Parquet/enhanced_summed_47photic.parquet", index=False)

print(df_final.head())

   ORD_OCC    CAST_ID         DATE_TIME_UTC         DATE_TIME_PST  LAT_DEC  \
0      1.0  9308_001d  1993-08-11T12:06:56Z  1993-08-11T04:06:56Z    -99.0   
1      6.0  9308_006d  1993-08-12T10:57:24Z  1993-08-12T02:57:24Z    -99.0   
2     11.0  9308_011d  1993-08-13T14:53:14Z  1993-08-13T06:53:14Z    -99.0   
3     14.0  9308_014d  1993-08-14T10:19:40Z  1993-08-14T02:19:40Z    -99.0   
4     18.0  9308_018d  1993-08-15T10:12:16Z  1993-08-15T02:12:16Z    -99.0   

   LON_DEC       STA_ID  LINE    STA  DEPTH  ...     Lon_Dec  Ac_Line  \
0    -99.0  093.3 026.7  93.3   26.7   29.0  ... -117.305000     93.3   
1    -99.0  093.3 045.0  93.3   45.0   47.0  ... -118.563333     93.3   
2    -99.0  093.3 080.0  93.3   80.0   59.0  ... -120.933333     93.3   
3    -99.0  093.3 110.0  93.3  110.0   77.0  ... -122.938333     93.2   
4    -99.0  090.0 100.0  90.0  100.0   58.0  ... -122.685000     89.9   

   Bottom_D  Secchi  IntChl  IntC14  TimeZone  Visibility     K_PAR  \
0      63.0    23.0  

In [8]:
# 1. How much of each metadata column is actually populated?
print(df[["Cruise_ID", "Cruz_Sta", "Cast_ID", "Secchi", "IntChl"]].isnull().sum())

# 2. Check if they use a fill value like 0, -9999, or empty string instead of NaN
print(df["Cruise_ID"].value_counts(dropna=False).head(10))

# 3. See what the raw data looks like for one cast
print(df[df["CAST_COUNT"] == df["CAST_COUNT"].iloc[0]][
    ["CAST_COUNT", "DEPTH", "Cruise_ID", "Secchi", "IntChl"]
])

Cruise_ID    1253192
Cruz_Sta     1253192
Cast_ID      1253192
Secchi       1253192
IntChl       1253203
dtype: int64
Cruise_ID
NaN                  1253192
2015-01-15-C-32NM         39
2014-07-06-C-32NM         37
2020-07-13-C-33SR         37
2009-07-14-C-31M4         37
2010-01-13-C-32NM         37
1996-08-07-C-32NM         37
2011-01-13-C-32NM         35
1995-07-06-C-31JD         35
1994-08-05-C-32NM         34
Name: count, dtype: int64
    CAST_COUNT  DEPTH          Cruise_ID  Secchi  IntChl
0        27453    2.0                NaN     NaN     NaN
1        27453    3.0                NaN     NaN     NaN
2        27453    4.0                NaN     NaN     NaN
3        27453    5.0                NaN     NaN     NaN
4        27453    6.0                NaN     NaN     NaN
5        27453    7.0                NaN     NaN     NaN
6        27453    8.0                NaN     NaN     NaN
7        27453    9.0                NaN     NaN     NaN
8        27453   10.0                NaN   